# 2.2 Demo - PII Data Security: Pseudonymization and Anonymization
**Lessons:** Pseudonymization & Anonymization · Summary & Best Practices

| Technique | Reversible? | Still personal data under GDPR? | Typical use |
|---|---|---|---|
| **Hashing** (salted / keyed) | No, but linkable | **Yes**, it is pseudonymized | Join keys across tables without exposing the identifier |
| **Tokenization** (lookup "vault") | Yes, for vault owners | **Yes** | Analytics on tokens, controlled re-identification |
| **Masking** | No | Yes, if other columns still identify the person | Display / support tools |
| **Generalization / binning** | No | No, *if* k-anonymity and similar checks hold | Sharing, ML features, open data |
| **Suppression** | No | No, with the same caveat | Remove direct identifiers and rare groups |
| **Perturbation (noise)** | No | No, with the same caveat | Aggregates, statistics |

Rule of thumb: **pseudonymization reduces risk, anonymization removes the data from GDPR scope**, but only if re-identification is truly impossible.

![Spectrum from identifiable raw PII, to pseudonymized data that is reversible with a secret, to anonymized data that is not reversible; GDPR applies to the first two](../images/2.2-privacy-spectrum.svg)

**How to read it:** sections A to C of this demo move data into the middle box, sections D and E into the right box. The GDPR bracket is the point of the table above: a hash or token is still personal data.

In [ ]:
import os
from adelab.classroom import setup
from adelab import datagen
from adelab.privacy import k_anonymity, equivalence_classes
from pyspark.sql import functions as F

DA = setup("2.2", reset=True)
datagen.land_users(f"{DA.paths.landing}/users")
users = (spark.read.json(f"{DA.paths.landing}/users")
         .withColumn("dob", F.to_date("dob")).withColumn("signup_ts", F.to_timestamp("signup_ts")))
users.write.saveAsTable("users_raw")
display(spark.table("users_raw").limit(5))

## A. Hashing with a secret salt (pseudonymization)
An unsalted `sha2(email)` is reversible in practice: an attacker can hash a list of known emails and compare. Add a **secret salt** (a pepper), stored in a secret scope and never in code:
```python
salt = dbutils.secrets.get(scope="privacy", key="pii_salt")   # on Databricks
```
Locally the salt comes from an environment variable, with a demo default.

In [ ]:
SALT = os.environ.get("ADELAB_PII_SALT", "demo-only-salt-change-me")

def pseudonymize(col):
    return F.sha2(F.concat(F.lit(SALT), F.lower(F.trim(col))), 256)

hashed = spark.table("users_raw").select(
    pseudonymize(F.col("email")).alias("user_key"),     # stable join key across systems
    "city", "region", "signup_ts")
display(hashed.limit(5))

The same input always produces the same key, so joins still work. Without the salt the keys can't be recomputed from a list of emails.

In [ ]:
events = spark.read.json(datagen.land_user_events(f"{DA.paths.landing}/events")).join(
    spark.table("users_raw").select("user_id", "email"), "user_id")
display(events.select(pseudonymize(F.col("email")).alias("user_key"), "event_type", "event_ts").limit(5))

## B. Tokenization with a vault
Replace each identifier with a random token and keep the mapping in a **separate, tightly controlled table** (the vault). Analysts work with tokens. Only an authorized process can re-identify. Deleting a person's vault row also breaks the link (*crypto-shredding*), which helps with deletion requests.

![Left: email plus a secret salt goes through sha2 to a one-way user_key. Right: email gets a random token stored in a restricted token_vault, and only the privacy team can join back from users_tokenized to the vault](../images/2.2-hashing-vs-tokenization.svg)

**How to read it:** the left side is section A (`pseudonymize()`), the right side is the next three cells. The red boxes are what you protect: the salt in a secret scope, the vault behind its own grants. The dashed red arrow is the re-identification query at the end of this section.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS token_vault (token STRING, email STRING, created_at TIMESTAMP);

MERGE INTO token_vault v
USING (SELECT DISTINCT email FROM users_raw) s
ON v.email = s.email
WHEN NOT MATCHED THEN INSERT (token, email, created_at) VALUES (uuid(), s.email, current_timestamp())

In [ ]:
%%sql
CREATE OR REPLACE TABLE users_tokenized AS
SELECT v.token AS customer_token, u.city, u.region, u.signup_ts
FROM users_raw u JOIN token_vault v ON u.email = v.email;

SELECT * FROM users_tokenized LIMIT 5

In [ ]:
%%sql
-- Re-identification: only the privacy team gets SELECT on token_vault
SELECT t.customer_token, v.email
FROM users_tokenized t JOIN token_vault v ON t.customer_token = v.token
LIMIT 3

## C. Masking
Built-in `mask()` replaces upper-case letters with `X`, lower-case letters with `x`, and digits with `n`. Add your own patterns with `regexp_replace`.

In [ ]:
display(spark.table("users_raw").select(
    "email",
    F.regexp_replace("email", r"(^.).*(@.*$)", "$1****$2").alias("email_masked"),
    "phone", F.expr("mask(phone)").alias("phone_masked"),
    "ssn", F.concat(F.lit("***-**-"), F.substring("ssn", -4, 4)).alias("ssn_masked")).limit(5))

## D. Anonymization: generalize, suppress, perturb
Even without names, **quasi-identifiers** can single a person out. About 87% of Americans are unique by (zip, birth date, gender). Generalize them:

In [ ]:
anonymized = (spark.table("users_raw")
    .withColumn("age", F.floor(F.months_between(F.lit("2026-10-15"), F.col("dob")) / 12))
    .select(
        # suppression: direct identifiers are removed entirely
        F.concat((F.floor(F.col("age") / 10) * 10).cast("int"), F.lit("-"),
                 (F.floor(F.col("age") / 10) * 10 + 9).cast("int")).alias("age_band"),   # generalization
        F.col("region"),                                                                # coarser than city/zip
        "gender",
        (F.round(F.col("salary") * (1 + (F.rand(seed=42) - 0.5) * 0.1), -3)).cast("long").alias("salary_noisy"),  # perturbation
        F.date_trunc("month", "signup_ts").alias("signup_month")))
anonymized.write.mode("overwrite").saveAsTable("users_anonymized")
display(spark.table("users_anonymized").limit(5))

## E. Measure re-identification risk: k-anonymity
A dataset is **k-anonymous** if every combination of quasi-identifiers is shared by at least *k* records.

![Raw zip, birth year and gender make every row unique (k = 1); generalizing to age band and region forms two classes of three, and suppressing the single 70-79 row gives k = 3](../images/2.2-k-anonymity.svg)

**How to read it:** a gray band is an equivalence class, and k is the size of the smallest one. `k_anonymity()` computes that number and `equivalence_classes()` lists the bands. The suppression cell after it drops the red rows, here with K = 5.

Compare the raw and the anonymized data:

In [ ]:
raw_with_age = spark.table("users_raw").withColumn("birth_year", F.year("dob"))
print("raw        k =", k_anonymity(raw_with_age, ["zip", "birth_year", "gender"]))
print("anonymized k =", k_anonymity(spark.table("users_anonymized"), ["age_band", "region", "gender"]))
display(equivalence_classes(spark.table("users_anonymized"), ["age_band", "region", "gender"]).limit(10))

Enforce a minimum *k* (for example 5) by **suppressing** rare groups before publishing:

In [ ]:
from pyspark.sql import Window

K = 5
qi = ["age_band", "region", "gender"]
published = (spark.table("users_anonymized")
             .withColumn("class_size", F.count("*").over(Window.partitionBy(*qi)))
             .filter(F.col("class_size") >= K).drop("class_size"))
print(f"published rows: {published.count()} of {spark.table('users_anonymized').count()}  |  k = {k_anonymity(published, qi)}")

## F. Best practices
- Classify first (demo 2.1), then choose a technique **per column** based on how the data is used.
- Pseudonymize as early as possible (Bronze → Silver), and keep raw PII only in a restricted Bronze with short retention.
- Keep salts, keys, and vaults in **secret scopes** and **separate catalogs** with separate grants.
- Pseudonymized data is still personal data. Deletion requests must reach it too (demo 2.3).
- Check k-anonymity (and l-diversity for sensitive attributes) before sharing "anonymous" data.

In [ ]:
DA.cleanup()